In [2]:
"""
Setup — runs once per notebook.
"""
import os
import json
from typing import Literal
from pydantic import BaseModel, Field, ValidationError
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not loaded"

client = Anthropic()
print("✓ Setup complete")

✓ Setup complete


In [1]:
from pydantic import BaseModel, Field
from typing import Literal
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()


class CodeSnippet(BaseModel):
    """A code snippet extracted from documentation."""
    language: Literal["python", "javascript", "bash", "json", "other"]
    code: str = Field(description="The actual code, no surrounding markdown fences")
    purpose: str = Field(description="One-sentence description of what this code does")


def extract_code_snippet(doc_text: str) -> CodeSnippet:
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        tools=[{
            "name": "save_snippet",
            "description": "Save a structured code snippet",
            "input_schema": CodeSnippet.model_json_schema(),
        }],
        tool_choice={"type": "tool", "name": "save_snippet"},
        messages=[{"role": "user", "content": f"Extract the code from this doc:\n\n{doc_text}"}],
    )
    for block in response.content:
        if block.type == "tool_use":
            return CodeSnippet(**block.input)
    raise ValueError("No tool use block in response")


sample_doc = """
To create a message with Claude, use the following Python code:
```python
from anthropic import Anthropic
client = Anthropic()
response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    messages=[{"role": "user", "content": "Hello!"}]
)
```
This sends a single user message and gets a response.
"""

snippet = extract_code_snippet(sample_doc)
print(snippet.model_dump_json(indent=2))

{
  "language": "python",
  "code": "from anthropic import Anthropic\nclient = Anthropic()\nresponse = client.messages.create(\n    model=\"claude-haiku-4-5-20251001\",\n    max_tokens=1024,\n    messages=[{\"role\": \"user\", \"content\": \"Hello!\"}]\n)",
  "purpose": "Create a message with Claude using the Anthropic API client"
}


In [3]:
"""
Exercise 2 — Extract a structured API method spec from documentation text.

We define two Pydantic models: APIParameter (the building block) and 
APIMethod (which contains a list of APIParameter). Claude returns the 
nested structure directly, no parsing needed.
"""

# Step 1: Define the nested schema
class APIParameter(BaseModel):
    """A single parameter of an API method."""
    name: str = Field(description="The parameter name, exactly as it appears in the docs")
    type: str = Field(description="The parameter type, e.g. 'string', 'integer', 'list[str]', 'dict'")
    required: bool = Field(description="True if the parameter is required, False if it has a default")
    description: str = Field(description="One-sentence description of what this parameter does")


class APIMethod(BaseModel):
    """A complete API method specification."""
    method_name: str = Field(description="The method name, e.g. 'messages.create'")
    purpose: str = Field(description="One-sentence description of what this method does")
    parameters: list[APIParameter] = Field(description="All parameters this method accepts")
    returns: str = Field(description="Description of what the method returns")


# Step 2: Extraction function
def extract_api_method(doc_text: str) -> APIMethod:
    """Send the doc to Claude and force it to return an APIMethod."""
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=2048,
        tools=[{
            "name": "save_api_method",
            "description": "Save a structured API method spec extracted from documentation",
            "input_schema": APIMethod.model_json_schema(),
        }],
        tool_choice={"type": "tool", "name": "save_api_method"},
        messages=[{"role": "user", "content": f"Extract the API method spec from this documentation:\n\n{doc_text}"}],
    )
    
    # Find the tool_use block in the response
    for block in response.content:
        if block.type == "tool_use":
            return APIMethod(**block.input)
    raise ValueError("Claude didn't call the save_api_method tool")


# Step 3: Test it with realistic API documentation
sample_doc = """
## messages.create

Create a Message with Claude. This is the primary endpoint for sending 
messages to a Claude model.

**Parameters:**

- `model` (string, required): The model identifier, e.g. "claude-haiku-4-5-20251001"
- `max_tokens` (integer, required): The maximum number of tokens to generate. 
  Must be a positive integer.
- `messages` (list, required): The conversation history. A list of message 
  objects with `role` and `content` fields.
- `system` (string, optional): A system prompt that provides context and 
  instructions to Claude. Defaults to no system prompt.
- `temperature` (float, optional): Controls randomness. Between 0.0 and 1.0. 
  Defaults to 1.0.
- `tools` (list, optional): A list of tool definitions the model can call. 
  Defaults to no tools.

**Returns:** A Message object containing the model's response, usage 
statistics, and stop reason.
"""

method = extract_api_method(sample_doc)

# Step 4: Inspect what Claude returned
print(f"Method: {method.method_name}")
print(f"Purpose: {method.purpose}")
print(f"\nParameters ({len(method.parameters)}):")
for param in method.parameters:
    required_marker = "required" if param.required else "optional"
    print(f"  - {param.name} ({param.type}, {required_marker}): {param.description}")
print(f"\nReturns: {method.returns}")

Method: messages.create
Purpose: Create a Message with Claude by sending messages to a Claude model.

Parameters (6):
  - model (string, required): The model identifier, e.g. "claude-haiku-4-5-20251001"
  - max_tokens (integer, required): The maximum number of tokens to generate, must be a positive integer.
  - messages (list, required): The conversation history as a list of message objects with role and content fields.
  - system (string, optional): A system prompt that provides context and instructions to Claude.
  - temperature (float, optional): Controls randomness, between 0.0 and 1.0.
  - tools (list, optional): A list of tool definitions the model can call.

Returns: A Message object containing the model's response, usage statistics, and stop reason.


In [4]:
# Variation 1: Try a different API doc — does it still extract cleanly?
sample_doc_2 = """
## get_weather(city: str, units: str = "fahrenheit") -> dict

Fetches the current weather for a given city. The city parameter must be 
a valid US city name. Units defaults to fahrenheit but can be set to 
celsius. Returns a dictionary with 'temp' (number), 'condition' (string), 
and 'humidity' (number).
"""
method_2 = extract_api_method(sample_doc_2)
print(method_2.model_dump_json(indent=2))


# Variation 2: Give Claude a doc that's not an API method — see if it 
# refuses gracefully or hallucinates a structure
nonsense_doc = """
The history of bread baking dates back thousands of years. Egyptians 
were among the first to discover that fermented dough...
"""
try:
    method_3 = extract_api_method(nonsense_doc)
    print(method_3.model_dump_json(indent=2))
except Exception as e:
    print(f"Failed: {e}")

{
  "method_name": "get_weather",
  "purpose": "Fetches the current weather for a given city.",
  "parameters": [
    {
      "name": "city",
      "type": "string",
      "required": true,
      "description": "A valid US city name."
    },
    {
      "name": "units",
      "type": "string",
      "required": false,
      "description": "Temperature units, either fahrenheit or celsius."
    }
  ],
  "returns": "A dictionary with 'temp' (number), 'condition' (string), and 'humidity' (number)."
}
{
  "method_name": "<UNKNOWN>",
  "purpose": "<UNKNOWN>",
  "parameters": [],
  "returns": "<UNKNOWN>"
}


""" the model returns UNKNOWN for nonsense prompt.
"""

In [ ]:
"""
Exercise 3 — Validation in action.

We define a model with strict constraints (ge=0 = "greater than or equal 
to zero"). Then we deliberately send Claude ambiguous input and see how 
Pydantic catches validation failures.
"""

# Step 1: Define a strict schema
class APIVersion(BaseModel):
    """A semantic version number."""
    major: int = Field(ge=0, description="Major version number — breaking changes")
    minor: int = Field(ge=0, description="Minor version number — new features, backward compatible")
    patch: int = Field(ge=0, description="Patch version number — bug fixes only")
    
    @property
    def version_string(self) -> str:
        """Computed property — not part of the schema sent to Claude."""
        return f"{self.major}.{self.minor}.{self.patch}"
    
    def is_stable(self) -> bool:
        """Convenience method — returns True if major version >= 1."""
        return self.major >= 1


# Step 2: Extraction function with explicit error handling
def extract_version(text: str) -> APIVersion | None:
    """
    Extract an APIVersion from text. Returns None and prints the error 
    if validation fails.
    """
    try:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=512,
            tools=[{
                "name": "save_version",
                "description": "Save the version number extracted from the text",
                "input_schema": APIVersion.model_json_schema(),
            }],
            tool_choice={"type": "tool", "name": "save_version"},
            messages=[{"role": "user", "content": f"Extract the version number from this text:\n\n{text}"}],
        )
        
        for block in response.content:
            if block.type == "tool_use":
                # Show what Claude actually returned, before validation
                print(f"Claude returned: {block.input}")
                # This is where validation happens
                return APIVersion(**block.input)
        
        print("No tool_use block found")
        return None
        
    except ValidationError as e:
        print(f"❌ Pydantic validation failed:")
        for err in e.errors():
            print(f"   Field: {err['loc']}, Error: {err['msg']}")
        return None


# Step 3: Test cases — easy, ambiguous, and adversarial
test_cases = [
    # Easy case — explicit version
    "We're upgrading to version 2.4.1 of the SDK.",
    
    # Ambiguous case — only major and minor
    "The release is version 3.1.",
    
    # Ambiguous case — text form
    "We just shipped the second major release.",
    
    # Adversarial case — invalid (negative would violate ge=0)
    "Rolling back to version -1.0.0 due to issues.",
    
    # No version mentioned at all
    "The product is amazing and our users love it.",
]

for text in test_cases:
    print(f"\n{'='*60}")
    print(f"Input: {text}")
    print('='*60)
    version = extract_version(text)
    if version:
        print(f"✓ Parsed: {version.version_string}")
        print(f"  Stable? {version.is_stable()}")


Input: We're upgrading to version 2.4.1 of the SDK.
Claude returned: {'major': 2, 'minor': 4, 'patch': 1}
✓ Parsed: 2.4.1
  Stable? True

Input: The release is version 3.1.
Claude returned: {'major': 3, 'minor': 1, 'patch': 0}
✓ Parsed: 3.1.0
  Stable? True

Input: We just shipped the second major release.
Claude returned: {'major': 2, 'minor': 0, 'patch': 0}
✓ Parsed: 2.0.0
  Stable? True

Input: Rolling back to version -1.0.0 due to issues.
Claude returned: {'major': -1, 'minor': 0, 'patch': 0}
❌ Pydantic validation failed:
   Field: ('major',), Error: Input should be greater than or equal to 0

Input: The product is amazing and our users love it.
Claude returned: {'major': 0, 'minor': 0, 'patch': 0}
✓ Parsed: 0.0.0
  Stable? False


: 

End-of-day reflection (notes/day1.md):

What surprised you about how this works?
How does Pydantic's JSON Schema differ from raw JSON Schema?
Where could the model still go wrong?